# Saccade Collection Pipeline

**Environment:** Activate the `eye_repo` conda environment and install the package (`pip install -e .`) before running this notebook.

This notebook curates saccade events across animals and recording blocks, compatible with the current **eye_tracking_system_tools** repository. It produces `all_saccade_collection` (a DataFrame of saccades with animal, block, eye, timestamps, and metrics) for downstream use (filtering by type, LFP extraction via `oe_rec.get_data()`, etc.).

**Prerequisites:**
- Block synchronization completed (`block_synchronization.ipynb`).
- `left_eye_data.csv` and `right_eye_data.csv` in each block's `analysis/` folder.
- `final_sync_df.csv` (or `blocksync_df.csv`) in each block's `analysis/` folder.

**Workflow:**
1. Build `block_collection` and `block_dict` from config (animals, block lists).
2. Load `final_sync_df` and eye data per block.
3. Run saccade detection per eye, attach `animal` / `block` / `eye`.
4. Split into synced (binocular) vs non-synced (monocular) events.
5. Build `all_saccade_collection = synced ∪ non_synced`.

*Electrophysiology (LFP) extraction will be added in a later step.*

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm

from eye_tracking_system_tools.preprocessing import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf

## Config

Set `experiment_path` (parent of animal folders), `animals`, `block_lists` (one list of block numbers per animal), and optional `bad_blocks`. Adjust `speed_threshold` and `diff_threshold` (ms) for saccade detection and L/R pairing.

In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")  # parent of animal folders
animals = ["PV_126","PV_106"]
block_lists = [[6],[15]]  # one list of block numbers per animal
bad_blocks = []
speed_threshold = 2.0   # saccade detection (speed_r > threshold)
diff_threshold_ms = 680  # max |t_L - t_R| (ms) to count as binocular pair

# Export: where to save all_saccade_collection (general analysis folder)
export_dir = experiment_path / "analysis" / "saccade_collections"
export_filename = "all_saccade_collection.csv"

## Block readiness check (optional)

Run the cell below to **input a list of animals** and scan all of their blocks for required files. This produces:
- **safe_blocks**: blocks that have degrees-raw-verified eye data, sync table, and behavioral state annotations (ready for the pipeline).
- **unsafe_blocks**: blocks that are missing one or more required files, with a list of what is missing for each.

Required files per block (in `block_path/analysis/`):
- **Eye data (degrees-raw-verified)**: `left_eye_data_degrees_raw_verified.csv`, `right_eye_data_degrees_raw_verified.csv` (with columns `OE_timestamp`, `center_x`, `center_y`, `ms_axis`).
- **Sync**: `final_sync_df.csv` or `blocksync_df.csv` (with columns `Arena_TTL`, `Arena_frame`, `L_eye_frame`, `R_eye_frame`, `L_values`, `R_values`).
- **Behavioral state**: `block_{block_num}_behavior_state.csv`.

You can then set `animals` and `block_lists` from the safe results before running the rest of the notebook.

In [ ]:
# -----------------------------------------------------------------------------
# Input: list of animals to check (edit list below; defaults to Config's `animals` if unset)
# -----------------------------------------------------------------------------
if "animals_to_check" not in dir():
    animals_to_check = list(animals)
# animals_to_check = ["PV_126", "PV_106"]  # optional: set explicitly to override

# Required file names and, for CSVs, required columns (None = only check existence).
REQUIRED_EYE_FILES = [
    "left_eye_data_degrees_raw_verified.csv",
    "right_eye_data_degrees_raw_verified.csv",
]
REQUIRED_EYE_COLUMNS = ["OE_timestamp", "center_x", "center_y", "ms_axis"]
SYNC_CANDIDATES = ["final_sync_df.csv", "blocksync_df.csv"]
REQUIRED_SYNC_COLUMNS = ["Arena_TTL", "Arena_frame", "L_eye_frame", "R_eye_frame", "L_values", "R_values"]
BEHAVIOR_STATE_PATTERN = "block_{block_num}_behavior_state.csv"  # block_num is 3-digit, e.g. 006


def discover_blocks_for_animals(experiment_path, animals):
    """Discover all (animal, date, block_num) by scanning experiment_path."""
    experiment_path = Path(experiment_path)
    out = []
    for animal in animals:
        p = experiment_path / animal
        if not p.exists():
            continue
        date_folders = [d for d in p.iterdir() if d.is_dir() and "block" not in str(d).lower()]
        for date_path in date_folders:
            for item in date_path.iterdir():
                if item.is_dir() and "block" in str(item).lower():
                    # block_006 -> 006
                    block_num = item.name[-3:] if len(item.name) >= 3 else item.name
                    try:
                        int(block_num)
                    except ValueError:
                        continue
                    out.append((animal, date_path.name, block_num))
    return out


def check_block_ready(analysis_path, block_num):
    """
    Check that all required files exist and (for CSVs) have required columns.
    Returns (is_ready, list_of_missing_or_invalid).
    """
    ap = Path(analysis_path)
    missing = []

    # Eye data (degrees-raw-verified)
    for fname in REQUIRED_EYE_FILES:
        p = ap / fname
        if not p.exists():
            missing.append(fname)
            continue
        try:
            df = pd.read_csv(p, index_col=0, engine="python", nrows=1)
            for c in REQUIRED_EYE_COLUMNS:
                if c not in df.columns:
                    missing.append(f"{fname} (missing column: {c})")
                    break
        except Exception as e:
            missing.append(f"{fname} (read error: {e})")

    # Sync file
    sync_path = None
    for name in SYNC_CANDIDATES:
        if (ap / name).exists():
            sync_path = ap / name
            break
    if sync_path is None:
        missing.append("sync file (final_sync_df.csv or blocksync_df.csv)")
    else:
        try:
            df = pd.read_csv(sync_path, nrows=1)
            for c in REQUIRED_SYNC_COLUMNS:
                if c not in df.columns:
                    missing.append(f"{sync_path.name} (missing column: {c})")
                    break
        except Exception as e:
            missing.append(f"sync file (read error: {e})")

    # Behavioral state
    behavior_name = BEHAVIOR_STATE_PATTERN.format(block_num=block_num)
    if not (ap / behavior_name).exists():
        missing.append(behavior_name)

    return (len(missing) == 0, missing)


# Discover all blocks and run checks
all_candidates = discover_blocks_for_animals(experiment_path, animals_to_check)
safe_blocks = []       # list of (animal, block_num) or block_key
unsafe_blocks = {}     # block_key -> list of missing files

for animal, date, block_num in all_candidates:
    block_key = f"{animal}_block_{block_num}"
    analysis_path = experiment_path / animal / date / f"block_{block_num}" / "analysis"
    if not analysis_path.exists():
        unsafe_blocks[block_key] = ["analysis/ folder missing"]
        continue
    ready, missing = check_block_ready(analysis_path, block_num)
    if ready:
        safe_blocks.append((animal, block_num))
    else:
        unsafe_blocks[block_key] = missing

# Build block_lists from safe_blocks: one list of block numbers per animal (for downstream use)
from collections import defaultdict
safe_by_animal = defaultdict(list)
for animal, block_num in safe_blocks:
    safe_by_animal[animal].append(int(block_num))
for k in safe_by_animal:
    safe_by_animal[k] = sorted(safe_by_animal[k])

# Summary
print("Block readiness check (degrees-raw-verified eye data + sync + behavior state)")
print("=" * 70)
print(f"Animals checked: {animals_to_check}")
print(f"Total blocks discovered: {len(all_candidates)}")
print(f"Safe blocks (ready for pipeline): {len(safe_blocks)}")
print(f"Unsafe blocks (missing files): {len(unsafe_blocks)}")
print()
if safe_blocks:
    print("Safe blocks (use these for animals + block_lists):")
    for animal in sorted(safe_by_animal.keys()):
        blks = safe_by_animal[animal]
        print(f"  {animal}: {blks}")
    print()
    print("To use only safe blocks, set in Config and re-run:")
    print("  animals =", list(safe_by_animal.keys()))
    print("  block_lists =", list(safe_by_animal.values()))
if unsafe_blocks:
    print()
    print("Unsafe blocks and missing files:")
    for block_key in sorted(unsafe_blocks.keys()):
        print(f"  {block_key}: {unsafe_blocks[block_key]}")

In [2]:
# Experiment path, animals, block_lists, and export paths are set in the Config cell above.
# Re-run that cell to change them; then run the block readiness check and the rest of the notebook.

## Helpers: block collection, load sync & eye data

- `create_block_collections`: builds `block_collection` (list of BlockSync) and `block_dict` with keys `"{animal}_block_{block_num}"`.
- `load_final_sync_df`: loads `final_sync_df.csv` or `blocksync_df.csv` into `block.final_sync_df` / `block.blocksync_df`.
- `load_eye_data`: loads `left_eye_data.csv` and `right_eye_data.csv` into `block.left_eye_data` / `block.right_eye_data`. Expects `OE_timestamp`, `center_x`, `center_y`, `ms_axis` (and optionally `pupil_diameter`).

In [3]:
def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """Build block_collection and block_dict from animals and block lists."""
    if bad_blocks is None:
        bad_blocks = []
    block_collection = []
    block_dict = {}
    for animal, blocks in zip(animals, block_lists):
        current = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks,

        )
        block_collection.extend(current)
        for b in current:
            block_dict[f"{animal}_block_{b.block_num}"] = b
    return block_collection, block_dict


def load_final_sync_df(block, filename=None, verbose=True):
    """Load final_sync_df from block.analysis_path and set block.final_sync_df / block.blocksync_df."""
    ap = Path(block.analysis_path)
    candidates = [filename] if filename else ["final_sync_df.csv", "blocksync_df.csv"]
    path = None
    for name in candidates:
        p = ap / name
        if p.exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"No sync file in {ap}. Tried: {candidates}")
    df = pd.read_csv(path)
    required = ["Arena_TTL", "Arena_frame", "L_eye_frame", "R_eye_frame", "L_values", "R_values"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")
    df = df.copy()
    df["Arena_TTL"] = df["Arena_TTL"].astype(float)
    if "ms_axis" not in df.columns and hasattr(block, "sample_rate") and block.sample_rate:
        df["ms_axis"] = df["Arena_TTL"] / (block.sample_rate / 1000)
    block.final_sync_df = df
    block.blocksync_df = df
    if verbose:
        print(f"[OK] Loaded {path.name} -> block.final_sync_df (rows={len(df):,})")
    return df


def load_eye_data(block, verbose=True):
    """Load left/right_eye_data from block.analysis_path. Expects OE_timestamp, center_x, center_y, ms_axis."""
    ap = Path(block.analysis_path)
    lp = ap / "left_eye_data.csv"
    rp = ap / "right_eye_data.csv"
    if not lp.exists() or not rp.exists():
        raise FileNotFoundError(f"Eye data not found in {ap}. Run block_synchronization pipeline first.")
    block.left_eye_data = pd.read_csv(lp, index_col=0, engine="python")
    block.right_eye_data = pd.read_csv(rp, index_col=0, engine="python")
    for name, df in [("left", block.left_eye_data), ("right", block.right_eye_data)]:
        for c in ["OE_timestamp", "center_x", "center_y", "ms_axis"]:
            if c not in df.columns:
                raise ValueError(f"{name}_eye_data missing column: {c}")
    if verbose:
        print(f"[OK] Loaded eye data for block {block.block_num} (L={len(block.left_eye_data)}, R={len(block.right_eye_data)})")

## Build block collection and load data

Instantiate blocks, load `final_sync_df` and eye data per block. Ensure each block has `sample_rate` (for `ms_axis`); it is set by BlockSync from the Open Ephys settings.

In [4]:
block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=bad_blocks,

)
for block in block_collection:
    load_final_sync_df(block)
    load_eye_data(block)
print(f"Blocks: {list(block_dict.keys())}")

instantiated block number 006 at Path: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006, new OE version
Found the sample rate for block 006 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\oe_files\PV126_Trial15_hunter7_2024-07-18_12-25-35\Record Node 102...
xml data matches file data.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 006
got it!
instantiated block number 015 at Path: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015, new OE version
Found the sample rate for block 015 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\oe_files\PV106_IMU_trial4_prey_2025-09-04_13-24-17\

## Saccade detection

`create_saccade_events_df` detects saccades from eye data (speed_r > threshold), computes onset/offset, magnitude, angle, and optional speed/diameter profiles. Uses `center_x`, `center_y`, `ms_axis`, `OE_timestamp`; `pupil_diameter` is optional.

In [5]:
def create_saccade_events_df(eye_data_df, speed_threshold, magnitude_calib=1.0, speed_profile=True, use_pupil_diameter=True):
    """
    Detect saccade events from eye tracking data (speed_r > threshold).
    Returns (df_with_speed, saccade_events_df). df is a copy with speed cols; original unchanged.
    """
    df = eye_data_df.copy()
    df["speed_x"] = df["center_x"].diff()
    df["speed_y"] = df["center_y"].diff()
    df["speed_r"] = (df["speed_x"] ** 2 + df["speed_y"] ** 2) ** 0.5
    df["is_saccade"] = df["speed_r"] > speed_threshold

    on_off = df["is_saccade"].astype(int) - df["is_saccade"].shift(periods=1, fill_value=False).astype(int)
    on_inds = np.where(on_off == 1)[0] - 1
    off_inds = np.where(on_off == -1)[0]
    if len(on_inds) == 0 or len(off_inds) == 0:
        ev = pd.DataFrame(columns=[
            "saccade_start_ind", "saccade_end_ind", "saccade_start_timestamp", "saccade_end_timestamp",
            "saccade_on_ms", "saccade_off_ms", "length", "magnitude_raw", "magnitude", "angle",
            "initial_x", "initial_y", "end_x", "end_y", "calib_dx", "calib_dy",
        ])
        if speed_profile:
            ev["speed_profile"] = []
        if use_pupil_diameter and "pupil_diameter" in df.columns:
            ev["diameter_profile"] = []
        df = df.drop(columns=["speed_x", "speed_y", "speed_r", "is_saccade"], errors="ignore")
        return df, ev

    on_ms = df["ms_axis"].iloc[on_inds].values
    on_ts = df["OE_timestamp"].iloc[on_inds].values
    off_ts = df["OE_timestamp"].iloc[off_inds].values
    off_ms = df["ms_axis"].iloc[off_inds].values

    ev = pd.DataFrame({
        "saccade_start_ind": on_inds,
        "saccade_end_ind": off_inds,
        "saccade_start_timestamp": on_ts,
        "saccade_end_timestamp": off_ts,
        "saccade_on_ms": on_ms,
        "saccade_off_ms": off_ms,
    })
    ev["length"] = ev["saccade_end_ind"] - ev["saccade_start_ind"]

    distances, angles, speed_list, diameter_list = [], [], [], []
    for _, row in tqdm.tqdm(ev.iterrows(), total=len(ev), desc="saccade metrics"):
        seg = df.loc[(df["OE_timestamp"] >= row["saccade_start_timestamp"]) &
                     (df["OE_timestamp"] <= row["saccade_end_timestamp"])]
        dist = seg["speed_r"].sum()
        distances.append(dist)
        if speed_profile:
            speed_list.append(seg["speed_r"].values)
        if use_pupil_diameter and "pupil_diameter" in df.columns:
            diameter_list.append(seg["pupil_diameter"].values)
        elif use_pupil_diameter:
            diameter_list.append(np.full(len(seg), np.nan))
        xi, yi = seg.iloc[0][["center_x", "center_y"]]
        xe, ye = seg.iloc[-1][["center_x", "center_y"]]
        ang = np.arctan2(ye - yi, xe - xi)
        angles.append(ang)

    ev["magnitude_raw"] = np.array(distances)
    ev["magnitude"] = np.array(distances) * magnitude_calib
    ev["angle"] = np.where(np.isnan(angles), np.nan, np.rad2deg(angles) % 360)
    start_ts = ev["saccade_start_timestamp"].values
    end_ts = ev["saccade_end_timestamp"].values
    start_df = df[df["OE_timestamp"].isin(start_ts)]
    end_df = df[df["OE_timestamp"].isin(end_ts)]
    ev["initial_x"] = start_df["center_x"].values
    ev["initial_y"] = start_df["center_y"].values
    ev["end_x"] = end_df["center_x"].values
    ev["end_y"] = end_df["center_y"].values
    ev["calib_dx"] = (ev["end_x"].values - ev["initial_x"].values) * magnitude_calib
    ev["calib_dy"] = (ev["end_y"].values - ev["initial_y"].values) * magnitude_calib
    if speed_profile:
        ev["speed_profile"] = speed_list
    if use_pupil_diameter and (("pupil_diameter" in df.columns) or diameter_list):
        ev["diameter_profile"] = diameter_list

    df = df.drop(columns=["speed_x", "speed_y", "speed_r", "is_saccade"], errors="ignore")
    return df, ev

## Per-block saccade extraction and collection

Run saccade detection on left and right eye data for each block, add `animal`, `block`, `eye`, then concatenate into `saccade_df_list` (one DataFrame per block, L+R stacked).

In [6]:
saccade_df_list = []
for block in block_collection:
    animal = block.animal_call
    blk = str(block.block_num)
    out = []
    for eye, attr in [("L", "left_eye_data"), ("R", "right_eye_data")]:
        eye_df = getattr(block, attr)
        _, sacc_ev = create_saccade_events_df(
            eye_df, speed_threshold,
            magnitude_calib=1.0, speed_profile=True, use_pupil_diameter="pupil_diameter" in eye_df.columns,
        )
        sacc_ev["animal"] = animal
        sacc_ev["block"] = blk
        sacc_ev["eye"] = eye
        out.append(sacc_ev)
    combined = pd.concat(out, ignore_index=True)
    saccade_df_list.append(combined)
saccade_collection = pd.concat(saccade_df_list, ignore_index=True)
print(f"Saccades per block: {[len(d) for d in saccade_df_list]}; total {len(saccade_collection)}")

saccade metrics: 100%|██████████| 210/210 [00:00<00:00, 1634.01it/s]


Saccades per block: [1950, 443]; total 2393


## Synced vs non-synced (binocular vs monocular)

`find_synced_saccades`: within each block, pair L and R saccades by `saccade_on_ms` (within `diff_threshold_ms`). Paired → synced (binocular); unpaired → non_synced (monocular). `combine_synced_dataframes` builds a single `synced_saccade_collection` with `Main`/`Sub` indices across blocks.

In [7]:
def find_synced_saccades(df, diff_threshold_ms=680, on_col="saccade_on_ms"):
    """Split saccades into synced (L–R pairs) and non_synced (unpaired). df has both eyes, animal, block."""
    l_df = df.query('eye == "L"').copy()
    r_df = df.query('eye == "R"').copy()
    synced_rows = []
    non_synced_rows = []

    for _, row in l_df.iterrows():
        t_l = row[on_col]
        dt = np.abs(r_df[on_col].values - t_l)
        ind = np.argmin(dt)
        if dt[ind] < diff_threshold_ms:
            synced_rows.append((row, r_df.iloc[ind]))
        else:
            non_synced_rows.append(row)

    r_matched = r_df.index.isin([r.index for _, r in synced_rows])
    r_leftovers = r_df.loc[~r_matched]

    n = len(synced_rows)
    idx = pd.MultiIndex.from_tuples(
        [(i, "L") for i in range(n)] + [(i, "R") for i in range(n)], names=["Main", "Sub"]
    )
    synced_df = pd.DataFrame(index=idx, columns=df.columns)
    for i, (l_row, r_row) in enumerate(synced_rows):
        synced_df.loc[(i, "L")] = l_row
        synced_df.loc[(i, "R")] = r_row

    non_synced_df = pd.concat([
        pd.DataFrame(non_synced_rows, columns=df.columns),
        r_leftovers,
    ], ignore_index=True)
    return synced_df, non_synced_df


def combine_synced_dataframes(synced_df_list):
    """Concatenate per-block synced DFs, reindex Main across blocks, reset_index."""
    out = []
    start = 0
    for sdf in synced_df_list:
        n = len(sdf) // 2
        if n == 0:
            continue
        idx = pd.MultiIndex.from_tuples(
            [(start + i, "L") for i in range(n)] + [(start + i, "R") for i in range(n)],
            names=["Main", "Sub"],
        )
        sdf = sdf.set_axis(idx)
        out.append(sdf)
        start += n
    if not out:
        return pd.DataFrame()
    combined = pd.concat(out)
    return combined.reset_index()

## Build `synced_saccade_collection`, `non_synced_saccade_collection`, `all_saccade_collection`

Run `find_synced_saccades` on each block's saccade DataFrame, then combine and concatenate.

In [8]:
synced_df_list = []
non_synced_df_list = []
for saccade_df in saccade_df_list:
    synced_df, non_synced_df = find_synced_saccades(
        saccade_df.dropna(subset=["saccade_on_ms"]), diff_threshold_ms=diff_threshold_ms, on_col="saccade_on_ms"
    )
    synced_df_list.append(synced_df)
    non_synced_df_list.append(non_synced_df)

synced_saccade_collection = combine_synced_dataframes(synced_df_list)
non_synced_saccade_collection = pd.concat(non_synced_df_list, ignore_index=True)
all_saccade_collection = pd.concat([synced_saccade_collection, non_synced_saccade_collection], ignore_index=True)

print(f"Synced: {len(synced_saccade_collection)}, non_synced: {len(non_synced_saccade_collection)}, all: {len(all_saccade_collection)}")

Synced: 2328, non_synced: 1229, all: 3557


## Label saccades with head movement (block_get_lizard_movement)

Use `block.block_get_lizard_movement()` to load movement data (`block.liz_mov_df` with columns `t_mov_ms`, `movAll`). A saccade is labeled `head_movement=True` only if at least one `t_mov_ms` value falls inside the original saccade interval: `[saccade_on_ms, saccade_off_ms]` (no temporal padding). If a block has no movement data (e.g. no `lizMov.mat`), those saccades remain `head_movement=False`. **Timebase:** `saccade_on_ms`/`saccade_off_ms` and `t_mov_ms` must be in the same units/reference (typically ms from block start).

In [12]:
def _block_key(animal, block):
    """Same key format as block_dict: animal_block_{block_num} with block as int."""
    b = str(block).strip()
    return f"{animal}_block_{int(b)}"


def _block_label(block_num):
    """Legacy-style block print label (e.g., block_007)."""
    return f"block_{int(block_num):03d}"


def _resolve_block_from_dict(block_dict, animal, block_num, debug_search=False):
    """
    Resolve block robustly across key formatting differences (e.g. 6 vs 006).
    Returns (resolved_key, block_obj or None).
    """
    candidates = [
        f"{animal}_block_{block_num}",
        f"{animal}_block_{str(block_num)}",
    ]

    try:
        b_int = int(str(block_num).strip())
        candidates.extend([
            f"{animal}_block_{b_int}",
            f"{animal}_block_{b_int:03d}",
        ])
    except Exception:
        b_int = None

    for k in candidates:
        if k in block_dict:
            return k, block_dict[k]

    if b_int is not None:
        prefix = f"{animal}_block_"
        for k, v in block_dict.items():
            if not str(k).startswith(prefix):
                continue
            tail = str(k)[len(prefix):]
            try:
                if int(str(tail).strip()) == b_int:
                    return k, v
            except Exception:
                continue

    if debug_search:
        print(f"[resolve] failed for animal={animal}, block={block_num}")
        print(f"[resolve] tried keys: {candidates}")
    return None, None


def _find_latest_lizmov_mat(block, block_label, debug_search=False):
    """
    Find lizMov.mat in nested Open Ephys layouts and return newest file.
    Debug mode prints every searched root and all discovered candidates.
    """
    from pathlib import Path

    roots = []
    if hasattr(block, "block_path") and block.block_path is not None:
        roots.append(Path(block.block_path) / "oe_files")
    if hasattr(block, "oe_path") and block.oe_path is not None:
        roots.append(Path(block.oe_path))
        roots.append(Path(block.oe_path).parent)

    unique_roots = []
    seen = set()
    for r in roots:
        r_str = str(r)
        if r_str not in seen:
            unique_roots.append(r)
            seen.add(r_str)

    if debug_search:
        print(f"[{block_label}] lizMov search roots:")
        for r in unique_roots:
            print(f"  - {r} (exists={r.exists()})")

    candidates = []
    for root in unique_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*.mat"):
                if p.name.lower() == "lizmov.mat":
                    candidates.append(p)
        except Exception as e:
            if debug_search:
                print(f"[{block_label}] failed to scan {root}: {e}")

    if debug_search:
        print(f"[{block_label}] found {len(candidates)} lizMov.mat candidates")
        for p in candidates:
            try:
                mtime = p.stat().st_mtime
            except OSError:
                mtime = -1
            print(f"    candidate: {p} (mtime={mtime})")

    if not candidates:
        return None

    preferred = [
        p for p in candidates
        if ("record node" in str(p).lower()) and ("analysis" in str(p).lower())
    ]
    pool = preferred if preferred else candidates

    try:
        newest = max(pool, key=lambda p: p.stat().st_mtime)
    except OSError:
        newest = pool[-1]

    if debug_search:
        print(f"[{block_label}] selected lizMov.mat: {newest}")
    return newest


def _load_lizard_movement_with_fallback(block, block_label, verbose=True, debug_search=False):
    """Try block method first, then recursive lizMov.mat fallback search."""
    print('hi, it is working')
    if hasattr(block, "liz_mov_df") and block.liz_mov_df is not None:
        if debug_search:
            print(f"[{block_label}] liz_mov_df already present on block")
        return True

    if debug_search:
        print(f"[{block_label}] trying block.block_get_lizard_movement()")
    try:
        block.block_get_lizard_movement()
    except Exception as e:
        if debug_search:
            print(f"[{block_label}] block_get_lizard_movement failed: {e}")

    if hasattr(block, "liz_mov_df") and block.liz_mov_df is not None:
        if verbose:
            print(f"{block_label} head movement file loaded")
        return True

    if debug_search:
        print(f"[{block_label}] trying recursive fallback search")
    mat_path = _find_latest_lizmov_mat(block, block_label, debug_search=debug_search)
    if mat_path is None:
        return False

    try:
        mat_data = h5py.File(str(mat_path), "r")
        acc_df = pd.DataFrame(
            data=np.array([mat_data["t_mov_ms"][:, 0], mat_data["movAll"][:, 0]]).T,
            columns=["t_mov_ms", "movAll"],
        )
        mat_data.close()
        block.liz_mov_df = acc_df
        if verbose:
            print(f"{block_label} head movement file loaded ({mat_path})")
        return True
    except Exception as e:
        if verbose:
            print(f"{block_label} fallback load error: {e}")
        return False


def add_head_movement_labels(saccade_df, block_dict, on_col="saccade_on_ms", off_col="saccade_off_ms", verbose=True, debug_search=False):
    """
    Legacy-aligned labeling: head_movement is True only when at least one t_mov_ms lies
    within the original saccade window [on_col, off_col] (no padding).
    """
    df = saccade_df.copy()
    required = ["animal", "block", on_col, off_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"saccade_df missing columns: {missing}")

    head_movement = np.zeros(len(df), dtype=bool)
    blocks_loaded = set()
    missing_or_failed_blocks = set()

    for i, row in enumerate(tqdm.tqdm(df.itertuples(index=False), total=len(df), desc="head_movement labels")):
        animal = getattr(row, "animal")
        block_num = getattr(row, "block")
        key, block = _resolve_block_from_dict(block_dict, animal, block_num, debug_search=debug_search)
        block_label = _block_label(block_num)

        if block is None:
            missing_key = f"{animal}_block_{block_num}"
            if missing_key not in missing_or_failed_blocks:
                print(f"{block_label} head movement file not found")
                if debug_search:
                    print(f"[{block_label}] block lookup miss in block_dict")
                missing_or_failed_blocks.add(missing_key)
            continue

        if key not in blocks_loaded:
            loaded = _load_lizard_movement_with_fallback(
                block,
                block_label,
                verbose=verbose,
                debug_search=debug_search,
            )
            if not loaded:
                print(f"{block_label} head movement file not found")
                missing_or_failed_blocks.add(key)
            blocks_loaded.add(key)

        if not (hasattr(block, "liz_mov_df") and block.liz_mov_df is not None):
            if key not in missing_or_failed_blocks:
                print(f"{block_label} head movement file not found")
                missing_or_failed_blocks.add(key)
            continue

        mov_times = block.liz_mov_df["t_mov_ms"].to_numpy()
        saccade_start = float(getattr(row, on_col))
        saccade_end = float(getattr(row, off_col))
        overlap = mov_times[(mov_times >= saccade_start) & (mov_times <= saccade_end)]
        if overlap.size > 0:
            head_movement[i] = True

    df["head_movement"] = head_movement
    if verbose:
        n_true = int(head_movement.sum())
        print(f"head_movement: True={n_true}, False={len(df) - n_true}")
    return df


all_saccade_collection = add_head_movement_labels(
    all_saccade_collection,
    block_dict,
    on_col="saccade_on_ms",
    off_col="saccade_off_ms",
    verbose=True,
    debug_search=True,
)

head_movement labels: 100%|██████████| 3557/3557 [00:00<00:00, 508214.31it/s]

block_006 head movement file not found
block_015 head movement file not found
head_movement: True=0, False=3557


## Export `all_saccade_collection`

Save `all_saccade_collection` to `export_dir` (general analysis folder). Optionally save a small manifest (animals, blocks, counts) as JSON for downstream pipelines.

In [ ]:
import json
from datetime import datetime

export_dir.mkdir(parents=True, exist_ok=True)
path_csv = export_dir / export_filename
all_saccade_collection.to_csv(path_csv, index=False)
print(f"[OK] Exported all_saccade_collection -> {path_csv} (rows={len(all_saccade_collection):,})")

n_synced_pairs = int(all_saccade_collection["Main"].dropna().nunique())
n_synced_rows = int(all_saccade_collection["Main"].notna().sum())
n_non_synced = int(all_saccade_collection["Main"].isna().sum())
manifest = {
    "export_time": datetime.now().isoformat(),
    "path": str(path_csv),
    "animals": sorted(all_saccade_collection["animal"].dropna().unique().astype(str).tolist()),
    "blocks": sorted(all_saccade_collection["block"].dropna().unique().astype(str).tolist()),
    "n_synced_pairs": n_synced_pairs,
    "n_synced_rows": n_synced_rows,
    "n_non_synced": n_non_synced,
    "n_total": len(all_saccade_collection),
}
path_manifest = export_dir / "all_saccade_collection_manifest.json"
with open(path_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"[OK] Manifest -> {path_manifest}")

## Summary

- **`block_collection`**: list of `BlockSync` objects.
- **`block_dict`**: `"{animal}_block_{block_num}"` → `BlockSync`.
- **`all_saccade_collection`**: DataFrame of all saccades (synced + non_synced) with `animal`, `block`, `eye`, `saccade_on_ms`, `saccade_off_ms`, `magnitude`, `angle`, etc. Synced rows have `Main` / `Sub`; non_synced do not.

Next steps (later): filter by type (e.g. `head_movement`, verified monocular), then LFP extraction via `block.oe_rec.get_data(...)`.

In [ ]:
all_saccade_collection.head()